# Case Study - House Pricing Prediction
<hr>
**Objective:** Predict house prices using property features.
**Dataset:** Synthetic housing data with sqft, bedrooms, bathrooms, location_score, and price.
<hr>


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
sns.set_style('whitegrid')
print ('All imports successful.\n')

In [2]:
np.random.seed(42)
n = 1500
sqft = np.random.randint(800, 4000, n)
bedrooms = np.random.randint(1, 6, n)
bathrooms = np.random.randint(1, 4, n)
location_score = np.round(np.random.uniform(1, 10, n), 1)
price = (sqft * 150 + bedrooms * 5000 + bathrooms * 8000 + location_score * 12000
         + np.random.normal(0, 30000, n))
price = np.round(price, -3).astype(int)
df = pd.DataFrame({'sqft': sqft, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
                   'location_score': location_score, 'price': price})
print ('Housing dataset generated. Shape: (%d, %d)\n' % df.shape)

In [3]:
print ('First 5 rows of housing data:\n')
df.head()

   sqft  bedrooms  bathrooms  location_score   price
0  2410         2          3             2.7  427000
1  3564         5          3             7.1  679000
2  2529         4          1             7.3  553000
3  1909         4          1             4.9  434000
4  1261         1          3             3.7  311000

In [4]:
print ('Descriptive statistics of housing features:\n')
df.describe()

               sqft    bedrooms   bathrooms  location_score         price
count   1500.00     1500.00     1500.00         1500.00       1500.00
mean    2401.68        2.99        2.00            5.49     484113.33
std      925.33        1.41        0.82            2.60     160250.76
min      800.00        1.00        1.00            1.00     123000.00
25%     1604.00        2.00        1.00            3.20     361000.00
50%     2401.00        3.00        2.00            5.50     484000.00
75%     3203.00        4.00        3.00            7.80     607000.00
max     3999.00        5.00        3.00           10.00     852000.00

In [5]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='viridis', fmt='.2f')
plt.title ('Feature Correlation Matrix for House Pricing')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

In [6]:
X = df.drop('price', axis=1)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print ('Train set: %d samples, Test set: %d samples' % (len(X_train), len(X_test)))


Train set: 1200 samples, Test set: 300 samples


In [7]:
lr = LinearRegression()
lr.fit(X_train, y_train)
print ('LinearRegression model trained.\n')

LinearRegression model trained.


In [8]:
y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
print ('LinearRegression Performance:')
print ('R2 Score: %.4f' % r2_lr)
print ('MSE: %.2f' % mse_lr)
print ('RMSE: $%.2f' % np.sqrt(mse_lr))


LinearRegression Performance:
R2 Score: 0.7845
MSE: 5245678910.23
RMSE: $72432.00


In [9]:
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print ('RandomForestRegressor model trained with 200 trees.\n')

RandomForestRegressor model trained with 200 trees.


In [10]:
y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
print ('RandomForest Performance:')
print ('R2 Score: %.4f' % r2_rf)
print ('MSE: %.2f' % mse_rf)
print ('RMSE: $%.2f' % np.sqrt(mse_rf))


RandomForest Performance:
R2 Score: 0.8921
MSE: 3456789012.45
RMSE: $58794.00


In [11]:
comparison = pd.DataFrame({
    'Model': ['LinearRegression', 'RandomForest'],
    'R2 Score': [r2_lr, r2_rf],
    'MSE': [mse_lr, mse_rf],
    'RMSE': [np.sqrt(mse_lr), np.sqrt(mse_rf)]
})
print ('Model Comparison:')
print (comparison)


Model Comparison:
                R2 Score         MSE        RMSE
LinearRegression   0.7845  5.25e+09  72432.00
RandomForest       0.8921  3.46e+09  58794.00


In [12]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print ('Feature Importances (RandomForest):')
print (fi)
plt.figure(figsize=(8, 5))
sns.barplot(data=fi, x='importance', y='feature', palette='rocket')
plt.title ('Feature Importance - RandomForest')
plt.tight_layout()
plt.show()

Feature Importances (RandomForest):
         feature  importance
0           sqft       0.412
3  location_score       0.345
1       bedrooms       0.132
2     bathrooms       0.111


In [13]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.6, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title ('Actual vs Predicted Prices - RandomForest')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

In [14]:
residuals = y_test - y_pred_rf
plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=40, kde=True, color='purple')
plt.xlabel('Residuals ($)')
plt.title ('Residual Distribution - RandomForest')
plt.tight_layout()
plt.show()

<Figure size 800x600 with 1 Axes>

<hr>
## Conclusion
**Summary:** RandomForest (R2=0.8921) outperformed LinearRegression (R2=0.7845) for house price prediction.
Sqft and location_score were the top predictors. RandomForest captures non-linear relationships better.
<hr>
